# Stage 2 — Topic Theme Map

Applies the annotated topic theme map to produce the final thematic breakdown.

**Runtime:** Google Colab + Google Drive  
**Storage:** All inputs and outputs are read from / written to a project folder on Google Drive.  
**Library:** [`multilingual-topic-modeling`](https://github.com/ay94/multilingual-topic-modeling)

**Inputs:**
- `annotated_messages.jsonl.gz` from Stage 1
- `topic_theme_map.csv` — manually produced by annotators (topic → sub-theme → theme)

**Outputs:**
- `themed_messages.jsonl.gz` — messages with theme and sub-theme assignments
- `evaluation_sample.csv` — sample for the evaluation stage

See [`docs/workflow/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/workflow) and [`docs/annotation/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/annotation) for the full methodology.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
!pip install multilingual-topic-modeling --quiet

In [ ]:
import numpy as np
import pandas as pd
from multilingual_topic import FileHandler

## Configuration

In [ ]:
# ── Edit these ────────────────────────────────────────────────────────────────
DRIVE_FOLDER  = '/content/drive/MyDrive/YOUR_PROJECT/Topic Modelling Workflow'
TOPIC_PREFIX  = 'topics'    # must match Stage 1
THEME_COL     = 'theme'
SUBTHEME_COL  = 'subtheme'
EVAL_SAMPLE_N = 500         # number of messages for the evaluation sample
# ──────────────────────────────────────────────────────────────────────────────

fh = FileHandler(DRIVE_FOLDER)

## 1. Load annotated messages and topic theme map

In [ ]:
annotated = pd.read_json(
    fh.create_filename('outputs/annotated_messages.jsonl.gz'),
    lines=True,
)
print(f'Annotated messages: {len(annotated):,}')

In [ ]:
# Topic theme map: produced by annotators
# Required columns: topic (int), subtheme (str), theme (str)
# Optionally: relevant (bool) — False marks a topic as irrelevant
topic_theme_map = pd.read_csv(fh.create_filename('inputs/topic_theme_map.csv'))
topic_theme_map.head()

## 2. Remove outliers and irrelevant topics

In [ ]:
# Drop -1 outliers
data = annotated[~annotated[f'{TOPIC_PREFIX}_outlier']].copy()

# Drop irrelevant topics if the column exists
if 'relevant' in topic_theme_map.columns:
    relevant_topics = topic_theme_map[topic_theme_map['relevant'] == True]['topic'].values
    data = data[data[TOPIC_PREFIX].isin(relevant_topics)]

print(f'After filtering: {len(data):,} messages')

## 3. Apply theme map

In [ ]:
# Standardise join column
topic_theme_map = topic_theme_map.rename(columns={'topic': TOPIC_PREFIX})
themed = data.merge(topic_theme_map[[TOPIC_PREFIX, SUBTHEME_COL, THEME_COL]], on=TOPIC_PREFIX, how='left')

print(themed[[THEME_COL, SUBTHEME_COL]].value_counts().head(20).to_string())

## 4. Save themed messages

In [ ]:
themed.to_json(
    fh.create_filename('outputs/themed_messages.jsonl.gz'),
    orient='records', lines=True,
)
print(f"Saved {len(themed):,} themed messages.")

## 5. Generate evaluation sample

Stratified sample for the evaluation stage. See [`docs/evaluation/`](https://github.com/ay94/topic-modeling-recipes/tree/main/docs/evaluation).

In [ ]:
eval_sample = themed.sample(n=min(EVAL_SAMPLE_N, len(themed)), random_state=1)
eval_sample.to_csv(fh.create_filename('outputs/evaluation_sample.csv'), index=False)
print(f'Evaluation sample: {len(eval_sample):,} rows')